03_preprocessing_and_feature_selection

In [1]:
#Load the V2 datasets

import pandas as pd
import numpy as np
import json

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SPLIT_DIR = PROJECT_ROOT / "fine_tuning" / "data_splits"

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "validation.csv")

print("Training:", train_df.shape)
print("Validation:", val_df.shape)

print("Target distribution:")
display(train_df["TARGET"].value_counts(normalize=True).round(4))

Training: (215257, 122)
Validation: (46127, 122)
Target distribution:


TARGET
0    0.9193
1    0.0807
Name: proportion, dtype: float64

In [2]:
# Cell 2: Separate features and target

TARGET_COL = "TARGET"
ID_COL = "SK_ID_CURR"

X_train_raw = train_df.drop(columns=[TARGET_COL, ID_COL])
y_train = train_df[TARGET_COL].copy()

X_val_raw = val_df.drop(columns=[TARGET_COL, ID_COL])
y_val = val_df[TARGET_COL].copy()

print("Training features:", X_train_raw.shape)
print("Validation features:", X_val_raw.shape)

print("Training target:", y_train.shape)
print("Validation target:", y_val.shape)

print("Target leakage check:",
      TARGET_COL not in X_train_raw.columns)

print("ID exclusion check:",
      ID_COL not in X_train_raw.columns)

Training features: (215257, 120)
Validation features: (46127, 120)
Training target: (215257,)
Validation target: (46127,)
Target leakage check: True
ID exclusion check: True


In [3]:
# Cell 3: Create reusable engineered features.
# Adds financial, demographic and external-score aggregates.
# Converts the employment sentinel value into missing data.

def create_engineered_features(data):
    df = data.copy()

    df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365.25

    employed_days = df["DAYS_EMPLOYED"].replace(365243, np.nan)
    df["EMPLOYED_YEARS"] = -employed_days / 365.25

    df["CREDIT_TO_INCOME"] = (
        df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )

    df["ANNUITY_TO_INCOME"] = (
        df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )

    df["CREDIT_TO_ANNUITY"] = (
        df["AMT_CREDIT"] / df["AMT_ANNUITY"].replace(0, np.nan)
    )

    df["GOODS_TO_CREDIT"] = (
        df["AMT_GOODS_PRICE"] / df["AMT_CREDIT"].replace(0, np.nan)
    )

    df["INCOME_PER_FAMILY_MEMBER"] = (
        df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"].replace(0, np.nan)
    )

    external_scores = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
    df["EXT_SOURCE_MEAN"] = external_scores.mean(axis=1)
    df["EXT_SOURCE_MAX"] = external_scores.max(axis=1)

    return df


X_train_fe = create_engineered_features(X_train_raw)
X_val_fe = create_engineered_features(X_val_raw)

print("Training features:", X_train_fe.shape)
print("Validation features:", X_val_fe.shape)

Training features: (215257, 129)
Validation features: (46127, 129)


In [5]:
# Cell 4: Separate columns by data type.
# Numerical columns will receive numerical imputation.
# Categorical columns will receive categorical imputation and encoding.

numeric_features = X_train_fe.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train_fe.select_dtypes(
    include=["object", "category", "bool", "str"]
).columns.tolist()

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numeric_features) + len(categorical_features))

Numerical features: 113
Categorical features: 16
Total features: 129


In [6]:
# Cell 5: Define separate preprocessing pipelines.
# Numerical data: median imputation with missingness indicators.
# Categorical data: most-frequent imputation and one-hot encoding.
# Unknown validation categories will be safely ignored.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    ))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


In [7]:
# Cell 6: Fit transformations exclusively on training data.
# Transform validation data using the already-fitted training pipeline.
# This prevents validation information from leaking into preprocessing.

X_train_processed = preprocessor.fit_transform(X_train_fe)
X_val_processed = preprocessor.transform(X_val_fe)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Output format:", type(X_train_processed).__name__)

Processed training shape: (215257, 321)
Processed validation shape: (46127, 321)
Output format: ndarray


In [8]:
# Cell 7: Extract transformed feature names.
# Identifies the original numerical and generated categorical columns.
# These names will help interpret feature importance and selection later.

processed_feature_names = preprocessor.get_feature_names_out()

print("Total transformed features:", len(processed_feature_names))
print("\nSample transformed features:")

for feature in processed_feature_names[:30]:
    print(feature)

Total transformed features: 321

Sample transformed features:
num__CNT_CHILDREN
num__AMT_INCOME_TOTAL
num__AMT_CREDIT
num__AMT_ANNUITY
num__AMT_GOODS_PRICE
num__REGION_POPULATION_RELATIVE
num__DAYS_BIRTH
num__DAYS_EMPLOYED
num__DAYS_REGISTRATION
num__DAYS_ID_PUBLISH
num__OWN_CAR_AGE
num__FLAG_MOBIL
num__FLAG_EMP_PHONE
num__FLAG_WORK_PHONE
num__FLAG_CONT_MOBILE
num__FLAG_PHONE
num__FLAG_EMAIL
num__CNT_FAM_MEMBERS
num__REGION_RATING_CLIENT
num__REGION_RATING_CLIENT_W_CITY
num__HOUR_APPR_PROCESS_START
num__REG_REGION_NOT_LIVE_REGION
num__REG_REGION_NOT_WORK_REGION
num__LIVE_REGION_NOT_WORK_REGION
num__REG_CITY_NOT_LIVE_CITY
num__REG_CITY_NOT_WORK_CITY
num__LIVE_CITY_NOT_WORK_CITY
num__EXT_SOURCE_1
num__EXT_SOURCE_2
num__EXT_SOURCE_3


In [9]:
# Cell 8: Train a preliminary XGBoost model for feature selection.
# Uses all transformed features to establish baseline importance.
# Validation data is used only for early stopping.



from xgboost import XGBClassifier

selection_model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)

selection_model.fit(
    X_train_processed,
    y_train,
    eval_set=[(X_val_processed, y_val)],
    verbose=False
)

print("Best iteration:", selection_model.best_iteration)
print("Best validation score:", selection_model.best_score)

Best iteration: 365
Best validation score: 0.25730807312698634


In [10]:
# Cell 9: Evaluate the preliminary model and rank original features.
# Aggregates encoded-column importance back to each source feature.

from sklearn.metrics import roc_auc_score, average_precision_score

# Validation performance
val_probabilities = selection_model.predict_proba(X_val_processed)[:, 1]

print("Validation ROC-AUC:", round(roc_auc_score(y_val, val_probabilities), 4))
print("Validation Average Precision:", round(average_precision_score(y_val, val_probabilities), 4))

# Map transformed columns back to their original features
importance_df = pd.DataFrame({
    "encoded_feature": processed_feature_names,
    "importance": selection_model.feature_importances_
})

def get_source_feature(encoded_name):
    name = encoded_name.split("__", 1)[-1]

    if name.startswith("missingindicator_"):
        name = name.replace("missingindicator_", "", 1)

    for feature in numeric_features + categorical_features:
        if name == feature or name.startswith(feature + "_"):
            return feature

    return name

importance_df["source_feature"] = importance_df["encoded_feature"].apply(
    get_source_feature
)

feature_importance = (
    importance_df.groupby("source_feature")["importance"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

display(feature_importance.head(20))

Validation ROC-AUC: 0.7652
Validation Average Precision: 0.2575


,source_feature,importance
0,ORGANIZATION_TYPE,0.138567
1,OCCUPATION_TYPE,0.054626
2,EXT_SOURCE_MEAN,0.047337
3,NAME_EDUCATION_TYPE,0.030267
4,NAME_INCOME_TYPE,0.025331
5,CODE_GENDER,0.022641
6,WEEKDAY_APPR_PROCESS_START,0.020872
7,NAME_HOUSING_TYPE,0.020428
8,NAME_FAMILY_STATUS,0.019116
9,NAME_TYPE_SUITE,0.018314


In [11]:
# Cell 10: Calculate SHAP values for the preliminary V2 model.
# Uses a validation sample to measure each feature's contribution to predictions.

import shap

X_shap = X_val_processed[:2000]

explainer = shap.TreeExplainer(selection_model)
shap_values = explainer.shap_values(X_shap)

print("SHAP values shape:", np.array(shap_values).shape)

c:\CODING\Credit_Risk\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP values shape: (2000, 321)


In [12]:
# Cell 11: Aggregate SHAP importance by original feature.
# Measures each original feature's average contribution magnitude.

shap_importance = pd.DataFrame({
    "encoded_feature": processed_feature_names,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
})

shap_importance["source_feature"] = (
    shap_importance["encoded_feature"].apply(get_source_feature)
)

shap_feature_ranking = (
    shap_importance.groupby("source_feature")["mean_abs_shap"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

display(shap_feature_ranking.head(20))

,source_feature,mean_abs_shap
0,EXT_SOURCE_MEAN,0.475902
1,CREDIT_TO_ANNUITY,0.152034
2,EXT_SOURCE_3,0.131987
3,CODE_GENDER,0.118991
4,GOODS_TO_CREDIT,0.117974
5,EMPLOYED_YEARS,0.093692
6,NAME_EDUCATION_TYPE,0.091969
7,EXT_SOURCE_MAX,0.085981
8,AMT_ANNUITY,0.078917
9,EXT_SOURCE_1,0.078253


In [14]:
# Cell 12: Check redundancy among numeric top-SHAP features.
# Excludes categorical columns because Spearman correlation requires numeric values.

numeric_top_features = [
    feature for feature in top_features
    if feature in numeric_features
]

correlation_matrix = X_train_fe[numeric_top_features].corr(method="spearman")

high_correlations = (
    correlation_matrix.where(
        np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
    )
    .stack()
    .reset_index()
)

high_correlations.columns = ["feature_1", "feature_2", "correlation"]
high_correlations["absolute_correlation"] = high_correlations["correlation"].abs()

display(
    high_correlations[
        high_correlations["absolute_correlation"] >= 0.80
    ].sort_values("absolute_correlation", ascending=False)
)

,feature_1,feature_2,correlation,absolute_correlation
179,AGE_YEARS,DAYS_BIRTH,-1.000000,1.000000
98,AMT_ANNUITY,AMT_GOODS_PRICE,0.828415,0.828415


In [15]:
# Cell 13: Define feature-selection candidates.
# Compares the full input set against a version without the redundant AGE_YEARS feature.

all_features = X_train_fe.columns.tolist()

candidate_features = {
    "full": all_features,
    "without_age_years": [
        feature for feature in all_features
        if feature != "AGE_YEARS"
    ]
}

for name, features in candidate_features.items():
    print(f"{name}: {len(features)} features")

full: 129 features
without_age_years: 128 features


In [16]:
# Cell 14: Compare full and reduced feature sets.
# Keeps model parameters and data partitions identical across experiments.

from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score

feature_experiment_results = []

for name, features in candidate_features.items():
    experiment_preprocessor = clone(preprocessor)

    numeric_subset = [f for f in numeric_features if f in features]
    categorical_subset = [f for f in categorical_features if f in features]

    experiment_preprocessor.set_params(
        transformers=[
            ("num", numeric_transformer, numeric_subset),
            ("cat", categorical_transformer, categorical_subset)
        ]
    )

    X_train_exp = experiment_preprocessor.fit_transform(X_train_fe[features])
    X_val_exp = experiment_preprocessor.transform(X_val_fe[features])

    experiment_model = clone(selection_model)
    experiment_model.fit(
        X_train_exp,
        y_train,
        eval_set=[(X_val_exp, y_val)],
        verbose=False
    )

    probabilities = experiment_model.predict_proba(X_val_exp)[:, 1]

    feature_experiment_results.append({
        "experiment": name,
        "input_features": len(features),
        "transformed_features": X_train_exp.shape[1],
        "best_iteration": experiment_model.best_iteration,
        "roc_auc": roc_auc_score(y_val, probabilities),
        "average_precision": average_precision_score(y_val, probabilities)
    })

display(pd.DataFrame(feature_experiment_results))

,experiment,input_features,transformed_features,best_iteration,roc_auc,average_precision
0,full,129,321,365,0.765156,0.257473
1,without_age_years,128,320,415,0.765606,0.256604


In [17]:
# Cell 15: Identify features with the lowest SHAP contribution.
# Measures how much of total SHAP importance the bottom-ranked features represent.

shap_feature_ranking["importance_share"] = (
    shap_feature_ranking["mean_abs_shap"]
    / shap_feature_ranking["mean_abs_shap"].sum()
)

bottom_features = shap_feature_ranking.tail(20).sort_values(
    "mean_abs_shap"
)

print(
    "Bottom 20 combined importance:",
    round(bottom_features["importance_share"].sum() * 100, 2),
    "%"
)

display(bottom_features.reset_index(drop=True))

Bottom 20 combined importance: 0.1 %


,source_feature,mean_abs_shap,importance_share
0,FLAG_MOBIL,0.000000,0.000000
1,FLAG_DOCUMENT_19,0.000000,0.000000
2,FLAG_DOCUMENT_20,0.000000,0.000000
3,FLAG_DOCUMENT_10,0.000000,0.000000
4,FLAG_DOCUMENT_4,0.000000,0.000000
5,FLAG_DOCUMENT_12,0.000000,0.000000
6,FLAG_DOCUMENT_17,0.000000,0.000000
7,FLAG_DOCUMENT_2,0.000012,0.000005
8,FLAG_DOCUMENT_7,0.000014,0.000005
9,FLAG_DOCUMENT_21,0.000021,0.000008


In [18]:
# Cell 16: Evaluate a reduced feature set.
# Removes the bottom 20 SHAP-ranked features and redundant AGE_YEARS.

low_impact_features = bottom_features["source_feature"].tolist()

reduced_features = [
    f for f in all_features
    if f not in low_impact_features and f != "AGE_YEARS"
]

reduced_numeric = [f for f in numeric_features if f in reduced_features]
reduced_categorical = [f for f in categorical_features if f in reduced_features]

reduced_preprocessor = clone(preprocessor)
reduced_preprocessor.set_params(transformers=[
    ("num", numeric_transformer, reduced_numeric),
    ("cat", categorical_transformer, reduced_categorical)
])

X_train_reduced = reduced_preprocessor.fit_transform(X_train_fe[reduced_features])
X_val_reduced = reduced_preprocessor.transform(X_val_fe[reduced_features])

reduced_model = clone(selection_model)
reduced_model.fit(
    X_train_reduced, y_train,
    eval_set=[(X_val_reduced, y_val)],
    verbose=False
)

reduced_probabilities = reduced_model.predict_proba(X_val_reduced)[:, 1]

reduced_result = {
    "experiment": "reduced_108",
    "input_features": len(reduced_features),
    "transformed_features": X_train_reduced.shape[1],
    "best_iteration": reduced_model.best_iteration,
    "roc_auc": roc_auc_score(y_val, reduced_probabilities),
    "average_precision": average_precision_score(y_val, reduced_probabilities)
}

display(pd.concat([
    pd.DataFrame(feature_experiment_results),
    pd.DataFrame([reduced_result])
], ignore_index=True))

,experiment,input_features,transformed_features,best_iteration,roc_auc,average_precision
0,full,129,321,365,0.765156,0.257473
1,without_age_years,128,320,415,0.765606,0.256604
2,reduced_108,108,296,378,0.764607,0.256358


In [19]:
# Cell 17: Save feature-selection results and candidate feature sets.
# Preserves the full and reduced configurations for model training in Notebook 04.

OUTPUT_DIR = PROJECT_ROOT / "fine_tuning" / "artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

feature_config = {
    "full_features": all_features,
    "reduced_features": reduced_features,
    "removed_features": sorted(set(all_features) - set(reduced_features)),
    "selection_method": "SHAP importance with controlled validation comparison",
    "reduced_feature_count": len(reduced_features),
    "full_feature_count": len(all_features)
}

with open(OUTPUT_DIR / "feature_selection_config.json", "w") as file:
    json.dump(feature_config, file, indent=4)

pd.DataFrame([
    *feature_experiment_results,
    reduced_result
]).to_csv(OUTPUT_DIR / "feature_experiment_results.csv", index=False)

shap_feature_ranking.to_csv(
    OUTPUT_DIR / "v2_shap_feature_ranking.csv",
    index=False
)

print("Feature-selection artifacts saved:", OUTPUT_DIR)

Feature-selection artifacts saved: c:\CODING\Credit_Risk\fine_tuning\artifacts
